# Arranque en Colab (opción A: subconjunto empaquetado en Drive)

**Sin ejecutar.** Cada sesión de Colab corre estas celdas en orden. Antes, en la PC con disco, se generaron y subieron a Drive los `.tar`
(`python scripts/empaquetar_subconjunto.py --config configs/paquete_subconjunto.yaml --salida <carpeta>`), con sus `.sha256`.

Completar las dos variables marcadas `# COMPLETAR` (no hay URL de repo ni carpeta de Drive fijadas en el proyecto).

In [ ]:
# 1) Montar Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2) Variables de la sesión
import os
DRIVE_DIR = "/content/drive/MyDrive/foliares"      # COMPLETAR: carpeta del equipo en Drive
REPO_URL  = "COMPLETAR_URL_DEL_REPO"                # COMPLETAR: p. ej. https://github.com/<org>/<repo>.git
REPO_REF  = "main"                                  # rama, tag o commit a usar (queda registrado abajo)
REPO_DIR  = "/content/foliares"
DATA_ROOT = "/content/data"                         # destino de los .tar (disco local de Colab, rápido)
PAQUETES_DRIVE = f"{DRIVE_DIR}/paquetes"            # plantvillage_subconjunto.tar(.sha256), plantdoc_subconjunto.tar(.sha256)
CKPT_DIR  = f"{DRIVE_DIR}/checkpoints"              # checkpoints y resultados: SIEMPRE en Drive (la sesión gratuita se corta)
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
# 3) Clonar el repo en el ref indicado
!git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout {REPO_REF}
%cd {REPO_DIR}

In [ ]:
# 4) Instalar el paquete (torch/torchvision ya vienen en Colab) y lo que falta
!pip install -q -e . --no-deps
!pip install -q imagehash pytest

In [ ]:
# 5) Copiar y descomprimir los .tar a DATA_ROOT (nombres dentro del tar = `ruta` de los manifiestos)
import shutil, tarfile, pathlib
os.makedirs(DATA_ROOT, exist_ok=True); os.makedirs("/content/paquetes", exist_ok=True)
tars = []
for ds in ("plantvillage", "plantdoc"):
    for ext in (".tar", ".tar.sha256"):
        shutil.copy(f"{PAQUETES_DRIVE}/{ds}_subconjunto{ext}", "/content/paquetes/")
    tars.append(f"/content/paquetes/{ds}_subconjunto.tar")
for t in tars:
    with tarfile.open(t) as tf:
        tf.extractall(DATA_ROOT, filter="data")
    print("descomprimido:", t)

In [ ]:
# 6) Fijar DATA_ROOT (lo lee configs/paths.yaml) y verificar sha256 del tar y de cada archivo
os.environ["DATA_ROOT"] = DATA_ROOT
!python scripts/verificar_paquete.py --tar {" ".join(tars)}
# Si imprime MAL en alguna línea, NO seguir: volver a copiar los .tar desde Drive.

In [ ]:
# 7) Las particiones congeladas viven en el repo (SPLITS_DIR), no en los .tar
from foliares.utils.paths import cargar_rutas
R = cargar_rutas()
print("DATA_ROOT:", R.data_root, "| SPLITS_DIR:", R.splits)
print(sorted(p.name for p in R.splits.iterdir()))
assert (R.splits / "README_VERSION.md").exists(), "Faltan las particiones congeladas en el repo: ¿se hizo el commit de data/splits/?"

In [ ]:
# 8) Registrar versión de torch, GPU y commit (guardado junto a los resultados, en Drive)
import json, platform, subprocess, time, torch
from foliares.utils.seeds import fijar_semillas
sesion = {
    "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
    "commit": subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip(),
    "repo_ref": REPO_REF,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_disponible": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data_root": DATA_ROOT,
    "semillas": fijar_semillas(42),
}
print(json.dumps(sesion, indent=2))
with open(f"{CKPT_DIR}/sesion_{time.strftime('%Y%m%d_%H%M%S')}.json", "w") as f:
    json.dump(sesion, f, indent=2)

## Checkpoints
`CKPT_DIR` (en Drive) es el directorio de checkpoints: guardar ahí al final de cada época y poder retomar. Registrar tiempo y memoria junto a cada métrica (CLAUDE.md).